# SageMaker Feature Store Initialization
This notebook handles the engineering and storage of features in the **AWS Feature Store**. This allows us to serve features for real-time predictions and keep a consistent history for training.

In [ ]:
!pip install "sagemaker<3.0.0"

import boto3
import sagemaker
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import pandas as pd
import numpy as np
import time
import sys
from time import gmtime, strftime

# Import our core engineering logic
sys.path.append('../src')
from preprocessing import run_prep
from features import run_engineering

# AWS Setup
sess = Session()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

sm_client = boto3.client("sagemaker", region_name=region)
fs_runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)

fs_session = Session(
    boto_session=boto3.Session(region_name=region),
    sagemaker_client=sm_client,
    sagemaker_featurestore_runtime_client=fs_runtime,
)

print(f"Role: {role}")

### Data Pipeline
Run the raw data through our engineering engine before pushing to the Feature Store.

In [ ]:
df = pd.read_csv('../data/raw/merged_sample.csv', low_memory=False)

# Core transformations
df = run_prep(df)
df = run_engineering(df)

# Feature Store requirements
df['EventTime'] = time.time()
df['collision_id'] = df['collision_id'].astype(str)

# Select the features we want to persist
cols = [
    'collision_id', 'EventTime', 'target', 
    'month', 'hour', 'is_rush_hour', 'is_weekend', 
    'cause_category', 'vehicle_type', 'borough'
]
features_df = df[cols].copy()

# Type casting for SageMaker SDK
for c in features_df.columns:
    if features_df[c].dtype == 'object':
        features_df[c] = features_df[c].astype('string')

print(f"Ready to ingest {features_df.shape[0]} records.")

### Feature Group Creation

In [ ]:
group_name = f"collisions-{strftime('%d-%H-%M', gmtime())}"

fg = FeatureGroup(name=group_name, sagemaker_session=fs_session)
fg.load_feature_definitions(data_frame=features_df)

fg.create(
    s3_uri=f"s3://{bucket}/aai-540-group6/offline-store/",
    record_identifier_name="collision_id",
    event_time_feature_name="EventTime",
    role_arn=role,
    enable_online_store=True
)

def wait_for_fg(group):
    while group.describe().get("FeatureGroupStatus") == "Creating":
        print("Creating...")
        time.sleep(5)
    print("Feature Group Ready.")

wait_for_fg(fg)

### Ingestion

In [ ]:
fg.ingest(data_frame=features_df, max_workers=3, wait=True)
print("Records pushed to online and offline stores.")

### Fetch a Sample Record
Verify retrieval from the online store.

In [ ]:
sample_id = features_df.iloc[0]['collision_id']
rec = fs_runtime.get_record(FeatureGroupName=group_name, RecordIdentifierValueAsString=sample_id)
display(rec.get('Record', 'Not Found'))